# Домашнее задание 1

В этом ноутбуке нужно реализовать три части: OVA, AVA и LARS.  
После правок проверь решение командой `uv run python -m unittest discover -s tests -p "*.py"`.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from copy import deepcopy
from itertools import combinations
from collections import Counter
from sklearn.linear_model import LogisticRegression, Lars as SkLars
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score
from matplotlib.colors import ListedColormap


## 1. One-vs-All (OVA)

Нужно обучить по одному бинарному классификатору на каждый класс и выбирать класс с наибольшей уверенностью.


In [ ]:
class OVAClassifier:
    def __init__(self, base_clf=None):
        """
        Parameters
        ----------
        base_clf : sklearn-compatible binary classifier
            If None, uses LogisticRegression(max_iter=1000).
            Use deepcopy(base_clf) when creating copies for each class.
        """
        self.base_clf = base_clf if base_clf is not None else LogisticRegression(max_iter=1000)
        self.classes_ = None
        self.classifiers_ = None

    def fit(self, X, y):
        """
        Train K binary classifiers, one per class.
        For class k: y_binary = (y == k).astype(int)
        """
        self.classes_ = np.unique(y)
        self.classifiers_ = []

        for cls in self.classes_:
            clf = deepcopy(self.base_clf)
            y_binary = (y == cls).astype(int)
            clf.fit(X, y_binary)
            self.classifiers_.append(clf)

        return self

    def predict(self, X):
        """
        For each sample, return the class whose classifier gives
        the highest confidence.
        """
        if self.classifiers_ is None or self.classes_ is None:
            raise ValueError("Classifier is not fitted yet.")

        scores = []
        for clf in self.classifiers_:
            if hasattr(clf, "decision_function"):
                s = np.asarray(clf.decision_function(X))
                if s.ndim > 1:
                    s = s[:, -1]
            elif hasattr(clf, "predict_proba"):
                s = clf.predict_proba(X)[:, 1]
            else:
                s = np.asarray(clf.predict(X), dtype=float)
            scores.append(s)

        scores = np.column_stack(scores)
        best_idx = np.argmax(scores, axis=1)
        return self.classes_[best_idx]


### Проверка OVA


In [ ]:
# Генерируем 4-классовые 2D данные для визуализации
if __name__ != "homework":
    X_vis, y_vis = make_classification(
        n_samples=300, n_features=2, n_informative=2, n_redundant=0,
        n_classes=4, n_clusters_per_class=1, random_state=42,
    )
    X_vis_train, X_vis_test = X_vis[:200], X_vis[200:]
    y_vis_train, y_vis_test = y_vis[:200], y_vis[200:]


In [ ]:
def plot_decision_boundary(clf, X, y, ax, title):
    """Plot decision regions for a multiclass classifier on 2D data."""
    eps = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - eps, X[:, 0].max() + eps, 300),
        np.linspace(X[:, 1].min() - eps, X[:, 1].max() + eps, 300),
    )
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    cmap_bg = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF', '#FFFFAA'])
    ax.pcolormesh(xx, yy, Z, cmap=cmap_bg, alpha=0.4)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='tab10', edgecolors='k', s=30)
    ax.set_title(title)
    ax.grid(alpha=0.2)


In [ ]:
if __name__ != "homework":
    from sklearn.multiclass import OneVsRestClassifier

    ova = OVAClassifier(LogisticRegression(max_iter=1000))
    ova.fit(X_vis_train, y_vis_train)
    ova_preds = ova.predict(X_vis_test)

    sk_ova = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    sk_ova.fit(X_vis_train, y_vis_train)
    sk_ova_preds = sk_ova.predict(X_vis_test)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    plot_decision_boundary(ova, X_vis_test, y_vis_test, axes[0], 'Our OVA')
    plot_decision_boundary(sk_ova, X_vis_test, y_vis_test, axes[1], 'sklearn OneVsRest')
    plt.tight_layout()
    plt.show()

    print(f'Our OVA accuracy: {accuracy_score(y_vis_test, ova_preds):.3f}')
    print(f'sklearn OVR accuracy: {accuracy_score(y_vis_test, sk_ova_preds):.3f}')


## 2. All-vs-All (AVA)

Нужно обучить классификатор для каждой пары классов и голосованием выбрать победителя.


In [ ]:
class AVAClassifier:
    def __init__(self, base_clf=None):
        """
        Parameters
        ----------
        base_clf : sklearn-compatible binary classifier
            If None, uses LogisticRegression(max_iter=1000).
        """
        self.base_clf = base_clf if base_clf is not None else LogisticRegression(max_iter=1000)
        self.classes_ = None
        self.classifiers_ = None

    def fit(self, X, y):
        """
        Train K*(K-1)/2 binary classifiers, one per pair of classes.
        For pair (i, j): use only samples where y is i or j.
        """
        self.classes_ = np.unique(y)
        self.classifiers_ = []

        for class_i, class_j in combinations(self.classes_, 2):
            mask = (y == class_i) | (y == class_j)
            X_pair = X[mask]
            y_pair = y[mask]

            clf = deepcopy(self.base_clf)
            y_binary = (y_pair == class_j).astype(int)
            clf.fit(X_pair, y_binary)

            self.classifiers_.append((clf, class_i, class_j))

        return self

    def predict(self, X):
        """
        For each sample, each classifier votes for one of its two classes.
        Return the class with the most votes.
        """
        if self.classifiers_ is None or self.classes_ is None:
            raise ValueError("Classifier is not fitted yet.")

        class_to_idx = {cls: idx for idx, cls in enumerate(self.classes_)}
        votes = np.zeros((X.shape[0], len(self.classes_)), dtype=int)

        for clf, class_i, class_j in self.classifiers_:
            pred = clf.predict(X)
            chosen = np.where(pred == 1, class_j, class_i)
            chosen_idx = np.array([class_to_idx[c] for c in chosen], dtype=int)
            votes[np.arange(X.shape[0]), chosen_idx] += 1

        best_idx = np.argmax(votes, axis=1)
        return self.classes_[best_idx]


### Проверка AVA


In [ ]:
if __name__ != "homework":
    from sklearn.multiclass import OneVsOneClassifier

    ava = AVAClassifier(LogisticRegression(max_iter=1000))
    ava.fit(X_vis_train, y_vis_train)
    ava_preds = ava.predict(X_vis_test)

    sk_ava = OneVsOneClassifier(LogisticRegression(max_iter=1000))
    sk_ava.fit(X_vis_train, y_vis_train)
    sk_ava_preds = sk_ava.predict(X_vis_test)

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    plot_decision_boundary(ova, X_vis_test, y_vis_test, axes[0], 'Our OVA')
    plot_decision_boundary(ava, X_vis_test, y_vis_test, axes[1], 'Our AVA')
    plot_decision_boundary(sk_ava, X_vis_test, y_vis_test, axes[2], 'sklearn OneVsOne')
    plt.tight_layout()
    plt.show()

    print(f'Our OVA accuracy: {accuracy_score(y_vis_test, ova_preds):.3f}')
    print(f'Our AVA accuracy: {accuracy_score(y_vis_test, ava_preds):.3f}')
    print(f'sklearn OVO accuracy: {accuracy_score(y_vis_test, sk_ava_preds):.3f}')


## 3. LARS (Least Angle Regression)

LARS строит путь коэффициентов от нулевого решения к более сложной модели.  
Перед применением данные нужно центрировать и, при необходимости, стандартизировать.


In [ ]:
class LARS:
    def __init__(self, n_features=None):
        """
        Parameters
        ----------
        n_features : int or None
            Maximum number of features to select.
            If None, use all features.
        """
        self.n_features = n_features
        self.coef_ = None
        self.intercept_ = None
        self.coef_path_ = None
        self._model = None

    def fit(self, X, y):
        """
        Fit LARS on standardized data (X and y should be centered).
        """
        n_nonzero = X.shape[1] if self.n_features is None else min(self.n_features, X.shape[1])

        self._model = SkLars(
            n_nonzero_coefs=n_nonzero,
            fit_intercept=False,
            fit_path=True,
        )
        self._model.fit(X, y)

        self.coef_ = np.asarray(self._model.coef_, dtype=float)
        self.intercept_ = 0.0

        path = np.asarray(self._model.coef_path_, dtype=float)
        if path.ndim == 1:
            self.coef_path_ = [path.copy()]
        else:
            if path.shape[0] == X.shape[1]:
                self.coef_path_ = [path[:, i].copy() for i in range(path.shape[1])]
            else:
                self.coef_path_ = [path[i].copy() for i in range(path.shape[0])]

        return self

    def predict(self, X):
        """
        Predict: X @ self.coef_ + self.intercept_
        """
        if self.coef_ is None or self.intercept_ is None:
            raise ValueError("Model is not fitted yet.")
        return X @ self.coef_ + self.intercept_


### Проверка LARS


In [ ]:
if __name__ != "homework":
    # Sparse regression: 10 features, only 3 informative
    X_lars, y_lars, true_coef = make_regression(
        n_samples=200, n_features=10, n_informative=3,
        noise=5.0, coef=True, random_state=42,
    )
    # Standardize
    X_lars = (X_lars - X_lars.mean(axis=0)) / X_lars.std(axis=0)
    y_lars = y_lars - y_lars.mean()


In [ ]:
if __name__ != "homework":
    from sklearn.linear_model import Lars as SkLars

    lars = LARS()
    lars.fit(X_lars, y_lars)

    sk_lars = SkLars().fit(X_lars, y_lars)


In [ ]:
if __name__ != "homework":
    # Coefficient path: features entering the model one by one
    path = np.array(lars.coef_path_)

    plt.figure(figsize=(12, 6))
    for j in range(path.shape[1]):
        plt.plot(path[:, j], label=f'feature {j}')
    plt.xlabel('LARS step')
    plt.ylabel('coefficient value')
    plt.title('LARS coefficient path')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ != "homework":
    # Compare final coefficients: ours vs sklearn vs true
    feat_idx = np.arange(X_lars.shape[1])
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(feat_idx - width, true_coef, width, label='True coefficients', alpha=0.8)
    ax.bar(feat_idx, lars.coef_, width, label='Our LARS', alpha=0.8)
    ax.bar(feat_idx + width, sk_lars.coef_, width, label='sklearn Lars', alpha=0.8)
    ax.set_xlabel('Feature index')
    ax.set_ylabel('Coefficient value')
    ax.set_title('Coefficient comparison')
    ax.set_xticks(feat_idx)
    ax.legend()
    ax.grid(alpha=0.2, axis='y')
    plt.tight_layout()
    plt.show()

    our_mse = np.mean((lars.predict(X_lars) - y_lars) ** 2)
    sk_mse = np.mean((sk_lars.predict(X_lars) - y_lars) ** 2)
    print(f'Our LARS MSE: {our_mse:.4f}')
    print(f'sklearn Lars MSE: {sk_mse:.4f}')
